# The 1970s Stagflation Stress Test

## Objective
Quantify the probability that the 2026 to 2030 macro regime will mimic the 1972 to 1982 stagnation template.

## Competing Narratives
- Hypothesis: the US is moving into a 1970s-style stagflation regime with persistent inflation, wage pressure, and energy-led supply shocks.
- Counter-hypothesis: the macro backdrop is better described as a volatile disinflation or sawtooth economy with short, sharp stress bursts rather than a flat decade-long inflation plateau.

In [1]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

try:
    import statsmodels.api as sm
except ImportError:
    sm = None

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from finance.config import load_env_file
from finance.data.clients.fred_api import fetch_fred_series

load_env_file()
FRED_API_KEY = os.getenv("FRED_API_KEY")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

if not FRED_API_KEY:
    warnings.warn(
        "FRED_API_KEY is not set. Public FRED pulls may still work for some series, but authenticated access is more reliable."
    )

print("Packages expected for the full standalone workflow: pandas, numpy, pandas_datareader, fredapi, yfinance, plotly, statsmodels, ipywidgets.")
print("This notebook uses the repo's FRED client first, so fredapi and pandas_datareader are optional rather than required.")

Packages expected for the full standalone workflow: pandas, numpy, pandas_datareader, fredapi, yfinance, plotly, statsmodels, ipywidgets.
This notebook uses the repo's FRED client first, so fredapi and pandas_datareader are optional rather than required.


/var/folders/_3/0gmvzdqs6zj5sc5vr52bhprr0000gn/T/ipykernel_34742/3107968504.py:41: UserWarning: FRED_API_KEY is not set. Public FRED pulls may still work for some series, but authenticated access is more reliable.
  warnings.warn(


In [11]:
START_DATE = "1970-01-01"

FRED_SERIES = {
    "CPI": "CPIAUCSL",
    "Wages": "ECIWAG",
    "GDP": "GDP",
    "FedFunds": "FEDFUNDS",
    "Debt_GDP": "GFDEGDQ188S",
    "Oil": "DCOILWTICO",
    "DXY": "DTWEXBGS",
    "Corp_Spread": "BAMLH0A0HYM2",
    "Productivity": "OPHNFB",
    "Services_CPI": "CUSR0000SASLE",
    "LFPR_55plus": "LNU01324230",
}

SERIES_FALLBACKS = {
    "Wages": ["AHETPI"],
    "DXY": ["DTWEXM"],
}

SERIES_RESAMPLING = {
    "CPI": "last",
    "Wages": "last",
    "GDP": "last",
    "FedFunds": "mean",
    "Debt_GDP": "last",
    "Oil": "mean",
    "DXY": "mean",
    "Corp_Spread": "mean",
    "Productivity": "last",
    "Services_CPI": "last",
    "LFPR_55plus": "mean",
}

QUARTERLY_FORWARD_FILL = {"Wages", "GDP", "Debt_GDP", "Productivity"}
COMPARISON_PERIODS = {
    "1973-1975": ("1973-01-31", "1975-12-31"),
    "1979-1982": ("1979-01-31", "1982-12-31"),
}

FRED_SERIES

{'CPI': 'CPIAUCSL',
 'Wages': 'ECIWAG',
 'GDP': 'GDP',
 'FedFunds': 'FEDFUNDS',
 'Debt_GDP': 'GFDEGDQ188S',
 'Oil': 'DCOILWTICO',
 'DXY': 'DTWEXBGS',
 'Corp_Spread': 'BAMLH0A0HYM2',
 'Productivity': 'OPHNFB',
 'Services_CPI': 'CUSR0000SASLE',
 'LFPR_55plus': 'LNU01324230'}

In [21]:
def _resample_series(series: pd.Series, how: str) -> pd.Series:
    if how == "mean":
        return series.resample("ME").mean()
    return series.resample("ME").last()


def _history_cutoff(label: str, start: str) -> pd.Timestamp:
    custom_cutoffs = {
        "Wages": pd.Timestamp(start) + pd.DateOffset(months=12),
        "DXY": pd.Timestamp("1974-01-31"),
    }
    return custom_cutoffs.get(label, pd.Timestamp(start))


def _has_enough_history(series: pd.Series, label: str, start: str) -> bool:
    first_valid = series.dropna().index.min()
    if pd.isna(first_valid):
        return False
    return first_valid <= _history_cutoff(label, start)


def _load_monthly_series(label: str, series_id: str, start: str) -> pd.Series:
    frame = fetch_fred_series(series_id, api_key=FRED_API_KEY, start=start)
    if frame.empty:
        raise RuntimeError("empty response")

    series = (
        frame.assign(observation_date=pd.to_datetime(frame["observation_date"]))
        .set_index("observation_date")
        [series_id]
        .sort_index()
    )
    monthly = _resample_series(series, SERIES_RESAMPLING.get(label, "last"))
    if label in QUARTERLY_FORWARD_FILL and series_id == FRED_SERIES[label]:
        monthly = monthly.ffill()
    return monthly


def fetch_monthly_dataset(
    series_map: dict[str, str], start: str = START_DATE
 ) -> tuple[pd.DataFrame, dict[str, str], dict[str, str]]:
    monthly_frames: list[pd.Series] = []
    errors: dict[str, str] = {}
    resolved_ids: dict[str, str] = {}

    for label, primary_series_id in series_map.items():
        candidate_ids = [primary_series_id, *SERIES_FALLBACKS.get(label, [])]
        candidate_series: list[pd.Series] = []
        loaded_ids: list[str] = []
        candidate_errors: list[str] = []

        for series_id in candidate_ids:
            try:
                candidate_series.append(_load_monthly_series(label, series_id, start))
                loaded_ids.append(series_id)
            except Exception as exc:
                candidate_errors.append(f"{series_id}: {exc}")

        if not candidate_series:
            errors[label] = "; ".join(candidate_errors) if candidate_errors else f"{primary_series_id}: unavailable"
            continue

        combined = candidate_series[0]
        for extra_series in candidate_series[1:]:
            combined = combined.combine_first(extra_series)

        if SERIES_FALLBACKS.get(label) and not _has_enough_history(combined, label, start):
            error_message = "; ".join(candidate_errors) if candidate_errors else "insufficient baseline history"
            errors[label] = error_message
            continue

        monthly_frames.append(combined.rename(label))
        resolved_ids[label] = " + ".join(loaded_ids)

    merged = pd.concat(monthly_frames, axis=1).sort_index() if monthly_frames else pd.DataFrame()
    merged.index.name = "Month"
    return merged, errors, resolved_ids


def score_to_verdict(score: float) -> str:
    if score <= 1:
        return "The plumbing rejects the 1970s thesis. Expect a Volatile Disinflationary Regime."
    if score <= 3:
        return "Moderate risk. Specific sectors are echoing the 1970s, but the macro aggregate is different."
    return "Model failure or regime change. Conditions match the 1970s transmission mechanism."


monthly, fetch_errors, resolved_series = fetch_monthly_dataset(FRED_SERIES, start=START_DATE)
display(pd.Series(resolved_series, name="resolved_series_id").to_frame())
display(monthly.tail())

if fetch_errors:
    display(pd.Series(fetch_errors, name="fetch_errors").to_frame())
else:
    print("All FRED series loaded successfully.")

,resolved_series_id
CPI,CPIAUCSL
Wages,ECIWAG + AHETPI
GDP,GDP
FedFunds,FEDFUNDS
Debt_GDP,GFDEGDQ188S
Oil,DCOILWTICO
DXY,DTWEXBGS + DTWEXM
Corp_Spread,BAMLH0A0HYM2
Productivity,OPHNFB
Services_CPI,CUSR0000SASLE


,CPI,Wages,GDP,FedFunds,Debt_GDP,Oil,DXY,Corp_Spread,Productivity,Services_CPI,LFPR_55plus
Month,,,,,,,,,,,
2025-12-31,326.031,31.83,NaN,3.72,NaN,57.972273,120.188323,2.891818,NaN,437.090,37.8
2026-01-31,326.588,31.94,NaN,3.64,NaN,60.037000,119.229810,2.740455,NaN,438.784,37.2
2026-02-28,327.460,32.02,NaN,3.64,NaN,64.508421,117.906021,2.919524,NaN,439.959,37.4
2026-03-31,330.293,32.07,NaN,3.64,NaN,91.383636,119.919855,3.187273,NaN,440.951,37.4
2026-04-30,NaN,NaN,NaN,NaN,NaN,104.821250,119.855513,2.980000,NaN,NaN,NaN


All FRED series loaded successfully.


## Feature Engineering: The Transmission Mechanisms

These cells translate raw macro time series into the mechanism-level variables that distinguish a true 1970s-style inflation spiral from a more cyclical disinflation regime.

In [22]:
df = monthly.copy()

df["Wages_YoY"] = df["Wages"].pct_change(12) * 100
df["CPI_YoY"] = df["CPI"].pct_change(12) * 100
df["Real_Wage_Growth"] = df["Wages_YoY"] - df["CPI_YoY"]

df[["Wages", "CPI", "Wages_YoY", "CPI_YoY", "Real_Wage_Growth"]].tail(18)

,Wages,CPI,Wages_YoY,CPI_YoY,Real_Wage_Growth
Month,,,,,
2024-11-30,170.500,316.528,3.710462,2.719472,0.990990
2024-12-31,170.500,317.604,3.710462,2.870691,0.839771
2025-01-31,171.800,318.961,3.369434,2.990978,0.378456
2025-02-28,171.800,319.679,3.369434,2.801583,0.567851
2025-03-31,171.800,319.785,3.369434,2.381981,0.987453
2025-04-30,173.566,320.302,3.559666,2.325388,1.234278
2025-05-31,173.566,320.620,3.559666,2.377265,1.182401
2025-06-30,173.566,321.435,3.559666,2.680454,0.879212
2025-07-31,174.954,322.169,3.584369,2.742618,0.841751


In [23]:
df["Nominal_GDP_YoY"] = df["GDP"].pct_change(12) * 100
df["Rate_Minus_Growth"] = df["FedFunds"] - df["Nominal_GDP_YoY"]

df[["FedFunds", "GDP", "Nominal_GDP_YoY", "Rate_Minus_Growth"]].tail(18)

,FedFunds,GDP,Nominal_GDP_YoY,Rate_Minus_Growth
Month,,,,
2024-11-30,4.64,29825.182,4.926908,-0.286908
2024-12-31,4.48,29825.182,4.926908,-0.446908
2025-01-31,4.33,30042.113,4.646595,-0.316595
2025-02-28,4.33,30042.113,4.646595,-0.316595
2025-03-31,4.33,30042.113,4.646595,-0.316595
2025-04-30,4.33,30485.729,4.592867,-0.262867
2025-05-31,4.33,30485.729,4.592867,-0.262867
2025-06-30,4.33,30485.729,4.592867,-0.262867
2025-07-31,4.33,31098.027,5.375376,-1.045376


In [24]:
df["Oil_Burden"] = (df["Oil"] / df["CPI"]) * 100
df["Oil_MoM"] = df["Oil"].pct_change()
df["DXY_MoM"] = df["DXY"].pct_change()
df["Oil_DXY_Corr"] = df["Oil_MoM"].rolling(24, min_periods=18).corr(df["DXY_MoM"])

df[["Oil", "CPI", "Oil_Burden", "Oil_DXY_Corr"]].tail(24)

,Oil,CPI,Oil_Burden,Oil_DXY_Corr
Month,,,,
2024-05-31,80.024545,313.175,25.552661,0.055255
2024-06-30,79.767368,313.044,25.481200,0.055900
2024-07-31,81.800455,313.569,26.086907,0.170585
2024-08-31,76.683182,314.062,24.416574,0.176409
2024-09-30,70.236000,314.732,22.316129,0.364972
2024-10-31,71.985000,315.631,22.806695,0.354010
2024-11-30,69.950000,316.528,22.099151,0.288225
2024-12-31,70.118095,317.604,22.077208,0.200010
2025-01-31,75.742500,318.961,23.746634,0.280624


In [25]:
df["Sticky_Services_YoY"] = df["Services_CPI"].pct_change(12) * 100

df[["Services_CPI", "Sticky_Services_YoY"]].tail(18)

,Services_CPI,Sticky_Services_YoY
Month,,
2024-11-30,423.277,4.591606
2024-12-31,424.227,4.448504
2025-01-31,426.204,4.313731
2025-02-28,427.422,4.112399
2025-03-31,427.911,3.677438
2025-04-30,429.192,3.576746
2025-05-31,429.914,3.532613
2025-06-30,431.105,3.611084
2025-07-31,432.617,3.655349


In [26]:
df["LFPR_55plus_Change"] = df["LFPR_55plus"].diff(12)

df[["LFPR_55plus", "LFPR_55plus_Change"]].tail(18)

,LFPR_55plus,LFPR_55plus_Change
Month,,
2024-11-30,38.5,-0.4
2024-12-31,38.3,-0.1
2025-01-31,38.3,0.0
2025-02-28,38.2,-0.4
2025-03-31,38.3,-0.4
2025-04-30,38.4,0.0
2025-05-31,38.1,-0.1
2025-06-30,37.7,-0.3
2025-07-31,37.9,-0.2


## Constructing the Similarity Index

The live score uses rolling 12 month averages for the key stress channels so the result is less sensitive to one-off monthly noise and better aligned with persistent regime behavior.

In [27]:
df["Real_Wage_Growth_12m"] = df["Real_Wage_Growth"].rolling(12, min_periods=12).mean()
df["Rate_Minus_Growth_12m"] = df["Rate_Minus_Growth"].rolling(12, min_periods=12).mean()
df["Oil_DXY_Corr_12m"] = df["Oil_DXY_Corr"].rolling(12, min_periods=12).mean()
df["Sticky_Services_YoY_12m"] = df["Sticky_Services_YoY"].rolling(12, min_periods=12).mean()
df["CPI_YoY_12m"] = df["CPI_YoY"].rolling(12, min_periods=12).mean()

signal_cols = [
    "Real_Wage_Growth_12m",
    "Rate_Minus_Growth_12m",
    "Oil_DXY_Corr_12m",
    "Sticky_Services_YoY_12m",
]
valid_signal_mask = df[signal_cols].notna().all(axis=1)

df["FLAG_WAGES"] = np.where(df["Real_Wage_Growth_12m"].notna(), (df["Real_Wage_Growth_12m"] > 1.5).astype(float), np.nan)
df["FLAG_RATES"] = np.where(df["Rate_Minus_Growth_12m"].notna(), (df["Rate_Minus_Growth_12m"] > 0.5).astype(float), np.nan)
df["FLAG_OIL"] = np.where(df["Oil_DXY_Corr_12m"].notna(), (df["Oil_DXY_Corr_12m"] < -0.3).astype(float), np.nan)
df["FLAG_SERVICES"] = np.where(df["Sticky_Services_YoY_12m"].notna(), (df["Sticky_Services_YoY_12m"] > 4.5).astype(float), np.nan)

flag_cols = ["FLAG_WAGES", "FLAG_RATES", "FLAG_OIL", "FLAG_SERVICES"]
df["Similarity_Score"] = df[flag_cols].sum(axis=1, min_count=len(flag_cols))

latest_idx = df.index[valid_signal_mask].max()
latest_snapshot = df.loc[latest_idx, [
    "Real_Wage_Growth_12m",
    "Rate_Minus_Growth_12m",
    "Oil_DXY_Corr_12m",
    "Sticky_Services_YoY_12m",
    *flag_cols,
    "Similarity_Score",
]]
latest_score = float(latest_snapshot["Similarity_Score"])
latest_verdict = score_to_verdict(latest_score)

display(latest_snapshot.to_frame(name=str(latest_idx.date())))
display(Markdown(f"**Latest Similarity Score:** {latest_score:.0f} / 4  \n**Verdict:** {latest_verdict}"))

,2025-09-30
Real_Wage_Growth_12m,0.853446
Rate_Minus_Growth_12m,-0.484603
Oil_DXY_Corr_12m,0.367115
Sticky_Services_YoY_12m,3.949123
FLAG_WAGES,0.000000
FLAG_RATES,0.000000
FLAG_OIL,0.000000
FLAG_SERVICES,0.000000
Similarity_Score,0.000000


**Latest Similarity Score:** 0 / 4  
**Verdict:** The plumbing rejects the 1970s thesis. Expect a Volatile Disinflationary Regime.

In [28]:
baseline_rows = []
for period_name, (start, end) in COMPARISON_PERIODS.items():
    window = df.loc[start:end, [*flag_cols, "Similarity_Score"]].dropna(how="all")
    baseline_rows.append({
        "period": period_name,
        "months": int(len(window)),
        "avg_score": float(window["Similarity_Score"].mean()) if not window.empty else np.nan,
        "max_score": float(window["Similarity_Score"].max()) if not window.empty else np.nan,
        "share_full_stagflation": float((window["Similarity_Score"] == 4).mean()) if not window.empty else np.nan,
        "wages_active_rate": float(window["FLAG_WAGES"].mean()) if not window.empty else np.nan,
        "rates_active_rate": float(window["FLAG_RATES"].mean()) if not window.empty else np.nan,
        "oil_active_rate": float(window["FLAG_OIL"].mean()) if not window.empty else np.nan,
        "services_active_rate": float(window["FLAG_SERVICES"].mean()) if not window.empty else np.nan,
    })

baseline = pd.DataFrame(baseline_rows).set_index("period")
scored_df = df.dropna(subset=["Similarity_Score"])
current_window = scored_df.loc[scored_df.index >= scored_df.index.max() - pd.DateOffset(months=36)]
current_baseline = pd.DataFrame({
    "period": ["Current 36M"],
    "months": [int(len(current_window))],
    "avg_score": [float(current_window["Similarity_Score"].mean())],
    "max_score": [float(current_window["Similarity_Score"].max())],
    "share_full_stagflation": [float((current_window["Similarity_Score"] == 4).mean())],
    "wages_active_rate": [float(current_window["FLAG_WAGES"].mean())],
    "rates_active_rate": [float(current_window["FLAG_RATES"].mean())],
    "oil_active_rate": [float(current_window["FLAG_OIL"].mean())],
    "services_active_rate": [float(current_window["FLAG_SERVICES"].mean())],
}).set_index("period")

pd.concat([baseline, current_baseline]).round(3)

,months,avg_score,max_score,share_full_stagflation,wages_active_rate,rates_active_rate,oil_active_rate,services_active_rate
period,,,,,,,,
1973-1975,36,NaN,NaN,0.0,0.25,0.306,NaN,0.667
1979-1982,48,NaN,NaN,0.0,0.00,0.729,NaN,1.000
Current 36M,37,0.973,2.0,0.0,0.00,0.000,0.108,0.865


In [29]:
plot_flags = df.loc["1990":, [*flag_cols, "Similarity_Score"]].dropna()
heatmap_labels = {
    "FLAG_WAGES": "Real Wages > 1.5%",
    "FLAG_RATES": "Rates Minus Growth > 0.5%",
    "FLAG_OIL": "Oil vs Dollar Correlation < -0.3",
    "FLAG_SERVICES": "Sticky Services > 4.5%",
}

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Heatmap(
        x=plot_flags.index,
        y=[heatmap_labels[col] for col in flag_cols],
        z=plot_flags[flag_cols].T.values,
        colorscale=[[0, "#1b7837"], [1, "#d73027"]],
        zmin=0,
        zmax=1,
        colorbar=dict(title="Flag"),
        hovertemplate="%{y}<br>%{x|%Y-%m}<br>Flag=%{z}<extra></extra>",
    ),
    secondary_y=False,
 )
fig.add_trace(
    go.Scatter(
        x=plot_flags.index,
        y=plot_flags["Similarity_Score"],
        mode="lines",
        line=dict(color="#1f1f1f", width=2),
        name="Similarity Score",
    ),
    secondary_y=True,
 )
fig.update_yaxes(title_text="Stress Channels", secondary_y=False)
fig.update_yaxes(title_text="Similarity Score", range=[-0.1, 4.1], secondary_y=True)
fig.update_layout(
    title="1970s Similarity Index Heatmap, 1990 to Present",
    xaxis_title="Time",
    template="plotly_white",
    height=600,
 )
fig.show()

In [30]:
def build_episode(series: pd.Series, start: str, end: str, label: str) -> pd.DataFrame:
    episode = series.loc[start:end].dropna().to_frame(name="Real_Wage_Growth")
    episode["Episode_Month"] = np.arange(1, len(episode) + 1)
    episode["Label"] = label
    return episode.reset_index()

episode_1974 = build_episode(df["Real_Wage_Growth"], "1974-01-31", "1974-12-31", "1974")
episode_now = build_episode(df["Real_Wage_Growth"], "2024-01-31", str(df.index.max().date()), "2024-2026")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=episode_1974["Episode_Month"],
        y=episode_1974["Real_Wage_Growth"],
        mode="lines",
        name="1974",
        line=dict(color="rgba(200, 0, 0, 0.6)", width=4),
    )
)
fig.add_trace(
    go.Scatter(
        x=episode_now["Episode_Month"],
        y=episode_now["Real_Wage_Growth"],
        mode="lines",
        name="2024-2026",
        line=dict(color="#1f77b4", width=3),
    )
)
fig.add_hline(y=0, line_dash="dash", line_color="#444")
fig.add_annotation(
    x=episode_now["Episode_Month"].max() if not episode_now.empty else 1,
    y=float(episode_now["Real_Wage_Growth"].min()) if not episode_now.empty else -1,
    text="Negative real wages in the 2020s are painful, but they are anti-stagflationary rather than self-reinforcing.",
    showarrow=False,
    xanchor="right",
    yshift=-25,
)
fig.update_layout(
    title="The Broken Spiral: Real Wage Growth Overlay",
    xaxis_title="Months Into Episode",
    yaxis_title="Real Wage Growth YoY",
    template="plotly_white",
)
fig.show()

In [31]:
plot_df = df[["FedFunds", "Nominal_GDP_YoY", "Rate_Minus_Growth"]].dropna()
positive = plot_df["Rate_Minus_Growth"] > 0
start_points = plot_df.index[positive & ~positive.shift(1, fill_value=False)]
end_points = plot_df.index[positive & ~positive.shift(-1, fill_value=False)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df["FedFunds"], name="Fed Funds", line=dict(color="#b22222", width=2)))
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df["Nominal_GDP_YoY"], name="Nominal GDP YoY", line=dict(color="#1f3b73", width=2)))

for start, end in zip(start_points, end_points):
    fig.add_vrect(x0=start, x1=end, fillcolor="rgba(200, 50, 50, 0.12)", line_width=0)

fig.add_annotation(
    x=pd.Timestamp("2024-06-30"),
    y=float(plot_df[["FedFunds", "Nominal_GDP_YoY"]].max().max()),
    text="1974 and 1980 saw rates exceed growth by far wider margins. In the 2020s the spread only briefly turned positive before easing.",
    showarrow=False,
    xanchor="left",
    yshift=20,
)
fig.update_layout(
    title="The Sawtooth Pattern: Fed Funds vs Nominal GDP Growth",
    xaxis_title="Time",
    yaxis_title="Percent",
    template="plotly_white",
)
fig.show()

In [32]:
energy_df = df[["Oil_DXY_Corr"]].dropna()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=energy_df.index,
        y=energy_df["Oil_DXY_Corr"],
        mode="lines",
        name="24M Oil vs DXY Correlation",
        line=dict(color="#8c510a", width=3),
    )
)
fig.add_hline(y=0, line_dash="dash", line_color="#444")
fig.add_annotation(
    x=energy_df.index.max(),
    y=float(energy_df["Oil_DXY_Corr"].iloc[-1]),
    text="1970s supply shocks pushed oil and the dollar in opposite directions. A more energy-independent US weakens that template today.",
    showarrow=True,
    arrowhead=2,
    ax=-160,
    ay=-40,
)
fig.update_layout(
    title="Energy Dominance: Oil vs Dollar Correlation",
    xaxis_title="Time",
    yaxis_title="Rolling Correlation",
    template="plotly_white",
)
fig.show()

## Model Verdict

Live verdict text is rendered by the scoring cell and the interactive scenario widget below. In a pure Jupyter notebook the markdown cell itself cannot update dynamically, so the interpretation rubric lives here while the computed verdict is displayed as output.

- Score 0 to 1: The plumbing rejects the 1970s thesis. Expect a Volatile Disinflationary Regime.
- Score 2 to 3: Moderate risk. Specific sectors are echoing the 1970s, but the macro aggregate is different.
- Score 4: Model failure or regime change. Conditions match the 1970s transmission mechanism.

In [33]:
latest_state = df.dropna(subset=["Similarity_Score"]).iloc[-1]
latest_dxy_trend = float(df["DXY"].pct_change(12).rolling(12, min_periods=12).mean().dropna().iloc[-1] * 100)

def scenario_score(oil_price: float, wage_growth_yoy: float) -> dict[str, float | int | str]:
    scenario_real_wage = wage_growth_yoy - float(latest_state["CPI_YoY_12m"])
    scenario_rate_minus_growth = float(latest_state["Rate_Minus_Growth_12m"])
    scenario_services = float(latest_state["Sticky_Services_YoY_12m"])

    scenario_oil_corr = float(latest_state["Oil_DXY_Corr_12m"])
    if oil_price >= 120 and latest_dxy_trend <= 0:
        scenario_oil_corr = min(scenario_oil_corr, -0.35)
    elif oil_price <= 80:
        scenario_oil_corr = max(scenario_oil_corr, 0.0)

    flags = {
        "FLAG_WAGES": int(scenario_real_wage > 1.5),
        "FLAG_RATES": int(scenario_rate_minus_growth > 0.5),
        "FLAG_OIL": int(scenario_oil_corr < -0.3),
        "FLAG_SERVICES": int(scenario_services > 4.5),
    }
    score = sum(flags.values())
    return {
        "real_wage_growth": scenario_real_wage,
        "oil_proxy_corr": scenario_oil_corr,
        "score": score,
        "verdict": score_to_verdict(score),
        **flags,
    }


if widgets is None:
    print("Install ipywidgets to enable the scenario slider cell.")
    display(pd.DataFrame([scenario_score(float(latest_state["Oil"]), float(latest_state["Wages_YoY"]))]).round(3))
else:
    oil_slider = widgets.FloatSlider(value=float(latest_state["Oil"]), min=40, max=160, step=1, description="Oil Price")
    wage_slider = widgets.FloatSlider(value=float(latest_state["Wages_YoY"]), min=0, max=8, step=0.1, description="Wage Growth")
    verdict_box = widgets.HTML()

    def refresh(*_args):
        result = scenario_score(oil_slider.value, wage_slider.value)
        verdict_box.value = (
            f"<b>Scenario Score:</b> {result['score']} / 4<br>"
            f"<b>Verdict:</b> {result['verdict']}<br>"
            f"<b>Real Wage Growth:</b> {result['real_wage_growth']:.2f}%<br>"
            f"<b>Oil-DXY Proxy Corr:</b> {result['oil_proxy_corr']:.2f}<br>"
            f"<b>Flags:</b> Wages={result['FLAG_WAGES']}, Rates={result['FLAG_RATES']}, Oil={result['FLAG_OIL']}, Services={result['FLAG_SERVICES']}"
        )

    oil_slider.observe(refresh, names="value")
    wage_slider.observe(refresh, names="value")
    refresh()
    display(
        widgets.VBox(
            [
                widgets.HTML("<b>Scenario Engine</b><br>This stress widget simplifies the oil flag by mapping very high oil prices plus a non-supportive dollar trend into a negative oil shock proxy."),
                oil_slider,
                wage_slider,
                verdict_box,
            ]
        )
    )

## Final Challenge to the Belief

The data argues for a sawtooth economy rather than a flat 1970s plateau of pain. The 1970s was a long grind of persistent inflationary transmission. The 2020s look more like alternating cliffs and rebounds. The investment strategy for one regime is survival. The investment strategy for the other is agility.